# 03 - Regime Detection

**Author:** Sacha Huberty

**Purpose:** Detect market regimes two ways -- an HMM on the broad-market
return/vol series (the primary, persistent risk switch) and K-Means on
FRED macro indicators (slower structural context) -- map both to named
postures/labels by profile, never by raw cluster index, then wire the
HMM posture into a V1 regime view: posture-conditional allocation-book
switching. This produces the project's first full strategy version and
backtests it OOS against the stage-2 baseline.

**Last updated:** 2026-07-25

## Setup

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.dates as mdates

pd.options.display.float_format = '{:.4f}'.format

from atlas import (
    allocation, backtest, data, metrics, regimes, strategy,
    strategy_legacy, universe,
)

cfg = data.load_config()
posture_cfg = data.load_config(data.PROJECT_ROOT / "config" / "regime_posture.yaml")
cfg["regimes"], posture_cfg["postures"].keys()

## Data

In [ ]:
as_of_universe = pd.Timestamp(cfg["general"]["is_end_date"])
universe_df = universe.load_universe(as_of_universe)
tickers = universe_df.index.tolist()
class_bucket = universe_df["class_bucket"]

prices = data.download_prices(tickers, start=cfg["general"]["start_date"])
prices = data.align_calendars(prices)
returns = data.daily_returns(prices)

market_ticker = cfg["regimes"]["market_ticker"]
market_returns = returns[market_ticker]
market_returns.tail()

In [ ]:
fred_series = cfg["regimes"]["kmeans_macro"]["fred_series"]
macro_df = data.download_macro(fred_series, start=cfg["general"]["start_date"])
macro_df.tail()

## Analysis / signal logic

### HMM market regime (primary switch)

Fit once across the full available history for this diagnostic chart
(an in-sample-style illustration, not a decision). The actual V1
strategy below refits the HMM weekly on trailing data only -- no
lookahead there.

In [ ]:
hmm_result = regimes.market_regime(market_returns, cfg, posture_cfg)
hmm_result["state_profiles"]

In [ ]:
print("State -> posture:", hmm_result["state_to_posture"])
transmat = hmm_result["transition_matrix"]
transmat.index = transmat.index.map(hmm_result["state_to_posture"])
transmat.columns = transmat.columns.map(hmm_result["state_to_posture"])
transmat

In [ ]:
posture_colors = {"risk_on": "#2a9d8f", "neutral": "#e9c46a", "risk_off": "#e76f51"}
posture_series = hmm_result["posture_series"]
aligned_prices = prices[market_ticker].reindex(posture_series.index)

fig, ax = plt.subplots(figsize=(12, 5))
ax.plot(aligned_prices.index, aligned_prices.values, color="black", linewidth=0.8)
for posture, color in posture_colors.items():
    mask = posture_series == posture
    ax.fill_between(
        posture_series.index, aligned_prices.min(), aligned_prices.max(),
        where=mask.values, color=color, alpha=0.25, label=posture,
    )
ax.set_title(f"{market_ticker} price, colored by HMM posture (primary switch)")
ax.set_ylabel("Price")
ax.legend(loc="upper left")
plt.tight_layout()
plt.show()

### Factor-style-by-regime (S4/S7)

Do equities really do better risk-on, and defensive assets better
risk-off? A quick sanity check on the same universe/bucket split used
throughout the project.

In [ ]:
bucket_returns = pd.DataFrame({
    bucket: returns[class_bucket[class_bucket == bucket].index].mean(axis=1)
    for bucket in class_bucket.unique()
})
by_posture = bucket_returns.reindex(posture_series.index).groupby(posture_series).mean() * 252
by_posture

### Macro K-Means (structural context)

Component 3's slower, structural regime signal. This stage treats it
as descriptive context only -- it is not yet wired into the V1
decision; it becomes a confidence input alongside the anomaly/GMM
agreement score in stage 4.

In [ ]:
elbow = regimes.macro_elbow_curve(macro_df, cfg, range(2, 8))
elbow.plot(marker="o", figsize=(7, 4), title="Macro K-Means: elbow curve")
plt.xlabel("k")
plt.ylabel("Inertia")
plt.show()

In [ ]:
macro_result = regimes.macro_kmeans(macro_df, cfg)
cluster_labels = regimes.label_macro_clusters(macro_df, macro_result["labels"], cfg)
print("Cluster -> business-cycle label:", cluster_labels)

coords = macro_result["pca_coords"].copy()
coords["cluster"] = macro_result["labels"].map(cluster_labels)
fig, ax = plt.subplots(figsize=(7, 6))
for label, group in coords.groupby("cluster"):
    ax.scatter(group["pc1"], group["pc2"], label=label, alpha=0.6, s=15)
ax.set_xlabel(f"PC1 ({macro_result['explained_variance_ratio'][0]:.1%} var)")
ax.set_ylabel(f"PC2 ({macro_result['explained_variance_ratio'][1]:.1%} var)")
ax.set_title("Macro K-Means clusters in PCA space")
ax.legend()
plt.show()

In [ ]:
macro_features = macro_result["features"]
heatmap_data = macro_features.groupby(macro_result["labels"]).mean()
heatmap_data.index = heatmap_data.index.map(cluster_labels)

fig, ax = plt.subplots(figsize=(8, 4))
im = ax.imshow(heatmap_data.values, cmap="RdBu_r", aspect="auto")
ax.set_xticks(range(len(heatmap_data.columns)))
ax.set_xticklabels(heatmap_data.columns, rotation=45, ha="right")
ax.set_yticks(range(len(heatmap_data.index)))
ax.set_yticklabels(heatmap_data.index)
ax.set_title("Regime-profile heatmap: mean YoY/12M-change by macro cluster")
fig.colorbar(im, ax=ax, label="value (standardization-scale units)")
plt.tight_layout()
plt.show()

### V1 wiring: posture switching -> first full strategy version

Each Friday: refit the HMM on trailing data only, pick the posture's
configured allocation book, and (for risk_off) apply the defensive
tilt. Unlike stage 2's classical books, the HMM cannot degrade
gracefully on a too-short window (it needs a full rolling-vol window
plus enough rows to tell states apart -- see
`regimes.has_enough_history`), so this backtest runs over the FULL
available history from `start_date`, falling back to the neutral
posture during the brief 2010 cold start. Every reported metric and
chart below still uses only OOS-dated returns.

In [ ]:
oos_start = pd.Timestamp(cfg["general"]["oos_start_date"])
lookback = cfg["optimization"]["lookback_days"]

regime_strategy_fn = strategy_legacy.regime_switching_strategy(class_bucket, cfg, posture_cfg)

regime_result_bt = backtest.run(regime_strategy_fn, returns, cfg)

In [ ]:
# Baselines from stage 2, recomputed here (same full-history run) for
# a direct, apples-to-apples comparison.
def make_classical_strategy(method):
    cov_method = cfg["optimization"]["covariance"]

    def strategy_fn(as_of, window):
        w = window.tail(lookback)
        cov_t = allocation.covariance_matrix(w, method=cov_method)
        if method == "risk_parity":
            return allocation.risk_parity(cov_t, cfg)
        raise ValueError(method)
    return strategy_fn


def permanent_strategy(as_of, window):
    return allocation.permanent(class_bucket)


baseline_fns = {
    "permanent": permanent_strategy,
    "risk_parity": make_classical_strategy("risk_parity"),
}
baseline_results = {
    name: backtest.run(fn, returns, cfg)
    for name, fn in baseline_fns.items()
}
all_results = {"regime_switching": regime_result_bt, **baseline_results}

## Results

In [ ]:
from atlas import metrics


def oos_metrics(result):
    r = result.daily_returns.loc[oos_start:]
    return {
        "ann_return": metrics.ann_return(r),
        "ann_vol": metrics.ann_vol(r),
        "sharpe": metrics.sharpe(r),
        "sortino": metrics.sortino(r),
        "calmar": metrics.calmar(r),
        "max_drawdown": metrics.max_drawdown(r),
        "hit_rate": metrics.hit_rate(r),
        "avg_weekly_turnover": result.turnover.loc[oos_start:].mean(),
        "total_cost_drag": result.costs.loc[oos_start:].sum(),
    }


comparison = pd.DataFrame(
    {name: oos_metrics(res) for name, res in all_results.items()}
).T
comparison.sort_values("sharpe", ascending=False)

In [ ]:
plt.figure(figsize=(11, 6))
for name, res in all_results.items():
    oos_curve = (1.0 + res.daily_returns.loc[oos_start:]).cumprod()
    oos_curve.plot(label=name, linewidth=2 if name == "regime_switching" else 1)
plt.title("OOS equity curves: V1 regime-switching vs. stage-2 baseline")
plt.ylabel("Growth of $1, rebased to OOS start")
plt.legend()
plt.show()

In [ ]:
oos_posture = posture_series.loc[oos_start:]
oos_posture.map({"risk_on": 0, "neutral": 1, "risk_off": 2}).plot(
    figsize=(11, 2.5), drawstyle="steps-post", title="HMM posture over the OOS window"
)
plt.yticks([0, 1, 2], ["risk_on", "neutral", "risk_off"])
plt.tight_layout()
plt.show()

## Notes / next steps

- **Honest OOS result: V1 alone does not yet beat the stage-2 baseline.** regime_switching scored Sharpe 0.75 OOS, vs. permanent's 1.10 and risk_parity's 0.84 (see the comparison table above). Expected at this point: only one naive view is active, the HMM found essentially a 2-regime split in practice (two states both nearest to risk_on, none nearest to neutral), and Black-Litterman -- which would calibrate this view's confidence against the others instead of switching allocation books wholesale -- does not arrive until stage 8. Per PROJECT_STRUCTURE.md's overfitting defense: an underperforming module gets iterated or deepened, or honestly documented as not earning its complexity -- not silently dropped or forced to look good.

- The HMM is the primary, persistent regime switch (S7/S8): refit
  weekly on a trailing 5-year window, states mapped to postures by
  their OWN empirical vol/return profile, never by raw state index --
  necessary since a re-fit can (and does) relabel which index means
  what.
- Macro K-Means is structural context only in this stage: it produces
  a genuine four-quadrant business-cycle read (Expansion/Slowdown/
  Contraction/Recovery) from FRED data, but does not yet feed the V1
  decision. It becomes a confidence input alongside the anomaly/GMM
  agreement score in stage 4.
- `risk_on`'s allocation_method is a naive `max_sharpe` on plain
  historical mu/cov for now, not the true `max_sharpe_bl` from
  `config/regime_posture.yaml`'s original intent -- stage 8's
  Black-Litterman fusion replaces it; until then, per
  PROJECT_STRUCTURE.md 5.1, "views can combine naively."
- `defensive_class_tilt` targets `fixed_income`/`commodity` (the
  actual universe.py bucket taxonomy), not the placeholder
  `short_bonds`/`gold` sub-labels the original config stub used --
  those sub-buckets don't exist anywhere in the screening pipeline.
- Next (stage 4): `anomaly.py` (sequential autoencoder + GMM latent
  regime), added as a risk override and as the missing confidence
  scaler alongside the HMM/macro agreement.